# CIC-IDS2017 — Network Intrusion Detection ML Pipeline
## Optimised for 16 GB RAM laptops (stratified 10 % sample + float32)

**Project:** Graduation Project — RL vs Static ML Firewall Comparison  
**Phase 1:** Train & test Random Forest + Gradient Boosting on CIC-IDS2017  
**Phase 2 (next notebook):** Test same models on NSL-KDD (concept-drift scenario)


## 1 · System Check

In [3]:
import platform, psutil, os

print("=" * 55)
print("  SYSTEM INFORMATION")
print("=" * 55)
print(f"  OS     : {platform.system()} {platform.release()}")
print(f"  CPU    : {os.cpu_count()} logical cores")
ram = psutil.virtual_memory()
print(f"  RAM    : {ram.total/1e9:.1f} GB total  |  {ram.available/1e9:.1f} GB free")
import subprocess, shutil
if shutil.which("nvidia-smi"):
    r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                       capture_output=True, text=True)
    print(f"  GPU    : {r.stdout.strip()}")
else:
    print("  GPU    : none detected (CPU only — fine for this pipeline)")
print("=" * 55)


  SYSTEM INFORMATION
  OS     : Windows 11
  CPU    : 8 logical cores
  RAM    : 16.9 GB total  |  5.0 GB free
  GPU    : NVIDIA GeForce MX350, 2048 MiB


## 2 · Imports

In [5]:
import pandas as pd
import numpy as np
import warnings, pickle, time, json, gc
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, matthews_corrcoef
)

import matplotlib
matplotlib.use('Agg')          # safe for all environments
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✓ All imports OK")


✓ All imports OK


## 3 · Data Loading — Stratified 10 % Sample

> **Why 10 %?**  
> The full CIC-IDS2017 dataset is ~3 million rows and needs >12 GB RAM to process.  
> A *stratified* sample keeps the exact same proportion of every attack class,  
> so the results are scientifically valid. Academic papers routinely do this  
> (see MDPI Adaptive RL paper in the project references).  
> At 10 % we still have ~280 000 rows — more than enough for robust ML.


In [8]:
# ── ① Change this path to where your CSV files live ──────────────────────────
DATA_DIR = Path(r'C:/Users/DELL/Desktop/Graduation Project/GP_Models/data/TrafficLabelling')
SAMPLE_FRAC = 0.10      # 10 % per class  (change to 0.20 if you have >20 GB RAM)
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────────────────────────────

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f.name}")

# Load every file, sample immediately, then concatenate
chunks = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, encoding='cp1252', low_memory=False)
        tmp.columns = tmp.columns.str.strip()
        
        # Identify label column
        label_col = None
        for candidate in ['Label', 'label', 'Class', 'class']:
            if candidate in tmp.columns:
                label_col = candidate
                break
        if label_col is None:
            label_col = tmp.columns[-1]
        
        # Stratified sample per file (preserves attack balance)
        sampled = (tmp.groupby(label_col, group_keys=False)
                      .apply(lambda x: x.sample(frac=SAMPLE_FRAC,
                                                 random_state=RANDOM_STATE)))
        chunks.append(sampled)
        del tmp
        gc.collect()
        print(f"  ✓ {f.name}: {len(sampled):,} rows sampled")
    except Exception as e:
        print(f"  ✗ {f.name}: {e}")

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"\n✓ Combined sampled dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


Found 8 CSV files:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv
  ✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 22,575 rows sampled
  ✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 28,647 rows sampled
  ✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 19,104 rows sampled
  ✓ Monday-WorkingHours.pcap_ISCX.csv: 52,992 rows sampled
  ✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 28,861 rows sampled
  ✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 17,037 rows sampled
  ✓ Tuesday-WorkingHours.pcap_ISCX.csv: 44,591 rows sampled
  ✓ Wednesday-workingHours.pcap_ISCX.csv: 69,270 rows sampled

✓ Combined sampled dataset: 283,07

## 4 · Memory Optimisation (float64 → float32)

In [10]:
# Halve RAM usage by downcasting numeric types
f64_cols = df.select_dtypes(include=['float64']).columns
i64_cols = df.select_dtypes(include=['int64']).columns

df[f64_cols] = df[f64_cols].astype(np.float32)
df[i64_cols] = df[i64_cols].astype(np.int32)

mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"✓ Memory usage after downcasting: {mem_mb:.1f} MB")

import psutil
avail_gb = psutil.virtual_memory().available / 1e9
print(f"✓ RAM still available: {avail_gb:.1f} GB")


✓ Memory usage after downcasting: 174.9 MB
✓ RAM still available: 5.1 GB


## 5 · Target Variable & Class Distribution

In [12]:
# Identify label column
LABEL_COL = None
for candidate in ['Label', 'label', 'Class', 'class']:
    if candidate in df.columns:
        LABEL_COL = candidate
        break
if LABEL_COL is None:
    LABEL_COL = df.columns[-1]

print(f"Label column: '{LABEL_COL}'")
print("\nClass counts:")
print(df[LABEL_COL].value_counts())

# Binary target  (0 = Benign, 1 = Attack)
df['target'] = (~df[LABEL_COL].str.strip().str.upper()
                               .isin(['BENIGN', 'NORMAL', 'LEGITIMATE'])).astype(np.int8)

print(f"\nBenign  (0): {(df['target']==0).sum():,}  "
      f"({(df['target']==0).mean()*100:.1f} %)")
print(f"Attack  (1): {(df['target']==1).sum():,}  "
      f"({(df['target']==1).mean()*100:.1f} %)")

# Quick plot
fig, ax = plt.subplots(figsize=(6, 3))
df['target'].value_counts().plot(kind='bar', color=['#2ecc71','#e74c3c'], ax=ax)
ax.set_xticklabels(['Benign (0)','Attack (1)'], rotation=0)
ax.set_title('Class Distribution (sampled dataset)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100)
plt.show()
print("✓ Plot saved: class_distribution.png")


Label column: 'Label'

Class counts:
Label
BENIGN                        227311
DoS Hulk                       23107
PortScan                       15893
DDoS                           12803
DoS GoldenEye                   1029
FTP-Patator                      794
SSH-Patator                      590
DoS slowloris                    580
DoS Slowhttptest                 550
Bot                              197
Web Attack – Brute Force         151
Web Attack – XSS                  65
Infiltration                       4
Web Attack – Sql Injection         2
Heartbleed                         1
Name: count, dtype: int64

Benign  (0): 227,311  (80.3 %)
Attack  (1): 55,766  (19.7 %)
✓ Plot saved: class_distribution.png


## 6 · Data Cleaning

In [15]:
# ── 6a  Standardise column names ────────────────────────────────────────────
df.columns = (df.columns
                .str.strip()
                .str.replace(' ', '_', regex=False)
                .str.replace(r'\xa0', '_', regex=True)
                .str.lower())

LABEL_COL = LABEL_COL.strip().lower().replace(' ','_')

# ── 6b  Remove infinite and NaN values ──────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
nan_before = df.isnull().sum().sum()
df[num_cols] = df[num_cols].fillna(0)
print(f"✓ Replaced {nan_before:,} NaN / Inf values with 0")

# ── 6c  Encode remaining object columns (e.g. protocol strings) ─────────────
obj_cols = [c for c in df.select_dtypes(include='object').columns
            if c not in [LABEL_COL, 'target']]
le_store = {}
for c in obj_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str)).astype(np.int32)
    le_store[c] = le
print(f"✓ Encoded {len(obj_cols)} categorical columns: {obj_cols}")

# ── 6d  Drop irrelevant columns ─────────────────────────────────────────────
drop_kw = ['timestamp','flowstarttime','flowendtime','src_ip','dst_ip',
           'sourceip','destip', LABEL_COL]
to_drop  = [c for c in df.columns
            if any(k in c for k in drop_kw) and c != 'target']
to_drop  = list(set(to_drop))
df.drop(columns=to_drop, inplace=True, errors='ignore')
print(f"✓ Dropped {len(to_drop)} irrelevant columns: {to_drop}")

print(f"\nFinal cleaned shape: {df.shape}")


✓ Replaced 550 NaN / Inf values with 0
✓ Encoded 4 categorical columns: ['flow_id', 'source_ip', 'destination_ip', 'timestamp']
✓ Dropped 2 irrelevant columns: ['label', 'timestamp']

Final cleaned shape: (283077, 84)


## 7 · Feature Preparation & Train / Test Split

In [17]:
X = df.drop(columns=['target']).values.astype(np.float32)
y = df['target'].values.astype(np.int64)
FEATURE_NAMES = df.drop(columns=['target']).columns.tolist()
FEATURE_DIM = X.shape[1]

print(f"Features: {FEATURE_DIM}")
print(f"Samples : {len(y):,}")

# ── Fix: drop exact duplicate rows BEFORE splitting ──────────────────────────
df_temp = pd.DataFrame(X)
df_temp['target'] = y
df_temp = df_temp.drop_duplicates()
X = df_temp.drop(columns=['target']).values.astype(np.float32)
y = df_temp['target'].values.astype(np.int64)
print(f"After dedup: {len(y):,} rows  (removed {len(df['target']) - len(y):,} duplicates)")
del df_temp

# ── Stratified split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# ── Verify no leakage ────────────────────────────────────────────────────────
train_set = set(map(tuple, X_train[:100].astype(np.float16)))
test_set  = set(map(tuple, X_test[:100].astype(np.float16)))
overlap   = train_set & test_set
print(f"Overlap check (should be 0): {len(overlap)}")

# ── Normalise ────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

X_train_sc = pd.DataFrame(X_train_sc, columns=FEATURE_NAMES)
X_test_sc  = pd.DataFrame(X_test_sc,  columns=FEATURE_NAMES)

del X
gc.collect()
print(f"\nTrain : {len(X_train_sc):,}  |  Test : {len(X_test_sc):,}")
print("✓ Split and scaling done")

Features: 83
Samples : 283,077
After dedup: 282,781 rows  (removed 296 duplicates)
Overlap check (should be 0): 0

Train : 197,946  |  Test : 84,835
✓ Split and scaling done


In [18]:
# Add this after your train/test split to verify no leakage
print(f"Train size: {len(X_train):,}")
print(f"Test size:  {len(X_test):,}")

# Check there's no overlap between train and test indices
# (should print 0 if clean)
train_set = set(map(tuple, X_train[:100]))
test_set  = set(map(tuple, X_test[:100]))
overlap   = train_set & test_set
print(f"Overlap in first 100 samples: {len(overlap)}  (should be 0)")

Train size: 197,946
Test size:  84,835
Overlap in first 100 samples: 0  (should be 0)


## 8 · Model 1 — Random Forest

In [20]:
print("Training Random Forest …")
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_sc, y_train)
rf_time = time.time() - t0

y_pred_rf    = rf.predict(X_test_sc)
y_proba_rf   = rf.predict_proba(X_test_sc)[:, 1]

print(f"\n✓ Random Forest trained in {rf_time:.1f} s")
print(f"  Accuracy : {(y_pred_rf == y_test).mean():.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_rf):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred_rf):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_proba_rf):.4f}")


Training Random Forest …

✓ Random Forest trained in 9.5 s
  Accuracy : 0.9997
  Precision: 0.9995
  Recall   : 0.9992
  F1-Score : 0.9993
  ROC-AUC  : 1.0000


## 9 · Model 2 — Gradient Boosting (baseline)

In [22]:
print("Training Gradient Boosting (baseline) …")
t0 = time.time()

gb_base = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
gb_base.fit(X_train_sc, y_train)
gb_base_time = time.time() - t0

y_pred_gb    = gb_base.predict(X_test_sc)
y_proba_gb   = gb_base.predict_proba(X_test_sc)[:, 1]

print(f"\n✓ GB baseline trained in {gb_base_time:.1f} s")
print(f"  Accuracy : {(y_pred_gb == y_test).mean():.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_gb):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_gb):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred_gb):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_proba_gb):.4f}")


Training Gradient Boosting (baseline) …

✓ GB baseline trained in 415.7 s
  Accuracy : 0.9999
  Precision: 0.9995
  Recall   : 0.9999
  F1-Score : 0.9997
  ROC-AUC  : 1.0000


## 10 · Hyperparameter Tuning — Gradient Boosting

> **Note:** Using `RandomizedSearchCV` instead of `GridSearchCV`.  
> It samples 20 random combinations instead of testing all 144 — roughly  
> **5× faster** with equally good results, which matters on a laptop.


In [24]:
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators'    : [100, 150, 200],
    'learning_rate'   : [0.05, 0.1, 0.15],
    'max_depth'       : [3, 5, 7],
    'min_samples_split': randint(5, 15),
    'subsample'       : uniform(0.7, 0.3)        # 0.7 – 1.0
}

print("Running RandomizedSearchCV (20 iterations × 3-fold CV) …")
print("This takes ~5–10 minutes on a laptop — please wait.\n")

rs = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

t0 = time.time()
rs.fit(X_train_sc, y_train)
tune_time = time.time() - t0

print(f"\n✓ Tuning done in {tune_time:.1f} s")
print(f"Best params : {rs.best_params_}")
print(f"Best CV F1  : {rs.best_score_:.4f}")


Running RandomizedSearchCV (20 iterations × 3-fold CV) …
This takes ~5–10 minutes on a laptop — please wait.

Fitting 3 folds for each of 20 candidates, totalling 60 fits

✓ Tuning done in 6556.9 s
Best params : {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 6, 'n_estimators': 150, 'subsample': 0.9684482051282945}
Best CV F1  : 0.9997


## 11 · Final Tuned Model — Evaluation

In [26]:
gb_final = rs.best_estimator_

y_pred_final  = gb_final.predict(X_test_sc)
y_proba_final = gb_final.predict_proba(X_test_sc)[:, 1]

print("=" * 55)
print("  FINAL MODEL PERFORMANCE (Test Set)")
print("=" * 55)
print(classification_report(y_test, y_pred_final,
                             target_names=['Benign','Attack']))

acc  = (y_pred_final == y_test).mean()
prec = precision_score(y_test, y_pred_final)
rec  = recall_score(y_test, y_pred_final)
f1   = f1_score(y_test, y_pred_final)
auc  = roc_auc_score(y_test, y_proba_final)
mcc  = matthews_corrcoef(y_test, y_pred_final)

print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {prec:.4f}")
print(f"  Recall      : {rec:.4f}")
print(f"  F1-Score    : {f1:.4f}")
print(f"  ROC-AUC     : {auc:.4f}")
print(f"  Matthews CC : {mcc:.4f}")


  FINAL MODEL PERFORMANCE (Test Set)
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     68128
      Attack       1.00      1.00      1.00     16707

    accuracy                           1.00     84835
   macro avg       1.00      1.00      1.00     84835
weighted avg       1.00      1.00      1.00     84835

  Accuracy    : 0.9998
  Precision   : 0.9995
  Recall      : 0.9998
  F1-Score    : 0.9996
  ROC-AUC     : 1.0000
  Matthews CC : 0.9995


## 12 · Visualisations

In [28]:
cm = confusion_matrix(y_test, y_pred_final)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('CIC-IDS2017 — Gradient Boosting (Tuned)', fontsize=14, fontweight='bold')

# 1 · Confusion matrix
ax = axes[0, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Benign','Attack'],
            yticklabels=['Benign','Attack'])
ax.set_title('Confusion Matrix')
ax.set_ylabel('True label'); ax.set_xlabel('Predicted label')

# 2 · ROC curve
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba_final)
ax.plot(fpr, tpr, lw=2, color='darkorange', label=f'AUC = {auc:.4f}')
ax.plot([0,1],[0,1], '--', color='navy', label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve')
ax.legend(); ax.grid(alpha=0.3)

# 3 · Precision–Recall curve
ax = axes[1, 0]
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba_final)
ax.plot(rec_curve, prec_curve, lw=2, color='green')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision–Recall Curve'); ax.grid(alpha=0.3)

# 4 · Model comparison bar chart
ax = axes[1, 1]
models  = ['Random Forest', 'GB Baseline', 'GB Tuned']
f1s     = [f1_score(y_test, y_pred_rf),
           f1_score(y_test, y_pred_gb),
           f1]
aucs    = [roc_auc_score(y_test, y_proba_rf),
           roc_auc_score(y_test, y_proba_gb),
           auc]
x = np.arange(len(models)); w = 0.35
ax.bar(x - w/2, f1s,  w, label='F1-Score', color='#3498db')
ax.bar(x + w/2, aucs, w, label='ROC-AUC',  color='#e74c3c')
ax.set_xticks(x); ax.set_xticklabels(models, rotation=10)
ax.set_ylim([0.85, 1.0]); ax.set_title('Model Comparison')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=100)
plt.show()
print("✓ Saved: model_evaluation.png")


✓ Saved: model_evaluation.png


## 13 · Feature Importance

In [30]:
fi = (pd.DataFrame({'feature': FEATURE_NAMES,
                         'importance': gb_final.feature_importances_})
        .sort_values('importance', ascending=False)
        .reset_index(drop=True))

print("Top 20 features:")
print(fi.head(20).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 bar
ax = axes[0]
top15 = fi.head(15)
ax.barh(range(15), top15['importance'], color='#27ae60')
ax.set_yticks(range(15)); ax.set_yticklabels(top15['feature'])
ax.invert_yaxis(); ax.set_xlabel('Importance')
ax.set_title('Top 15 Features'); ax.grid(alpha=0.3, axis='x')

# Cumulative importance
ax = axes[1]
cum = fi['importance'].cumsum() / fi['importance'].sum() * 100
ax.plot(range(1, len(cum)+1), cum, lw=2)
ax.axhline(80, color='r',      ls='--', label='80 %')
ax.axhline(90, color='orange', ls='--', label='90 %')
ax.set_xlabel('# Features'); ax.set_ylabel('Cumulative importance (%)')
ax.set_title('Cumulative Feature Importance'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()

n80 = int((cum <= 80).sum()) + 1
n90 = int((cum <= 90).sum()) + 1
print(f"\n{n80} features → 80 % importance")
print(f"{n90} features → 90 % importance  (out of {len(FEATURE_NAMES)} total)")


Top 20 features:
                feature  importance
              source_ip    0.846533
            source_port    0.147579
         destination_ip    0.002530
                flow_id    0.001278
       destination_port    0.000953
          bwd_packets/s    0.000581
 init_win_bytes_forward    0.000165
            bwd_iat_min    0.000072
            fwd_iat_min    0.000046
  bwd_packet_length_max    0.000022
           flow_iat_min    0.000020
 fwd_packet_length_mean    0.000017
    average_packet_size    0.000016
             active_min    0.000016
init_win_bytes_backward    0.000016
   avg_fwd_segment_size    0.000015
      packet_length_std    0.000011
 bwd_packet_length_mean    0.000009
 packet_length_variance    0.000008
           flow_iat_std    0.000008

1 features → 80 % importance
2 features → 90 % importance  (out of 83 total)


## 14 · Save Model Artefacts

In [32]:
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

# Models
with open(OUT/'rf_model.pkl',    'wb') as fh: pickle.dump(rf,       fh)
with open(OUT/'gb_final.pkl',    'wb') as fh: pickle.dump(gb_final, fh)
with open(OUT/'scaler.pkl',      'wb') as fh: pickle.dump(scaler,   fh)
with open(OUT/'feature_names.pkl','wb') as fh: pickle.dump(FEATURE_NAMES, fh)

# Feature importance CSV
fi.to_csv(OUT/'feature_importance.csv', index=False)

# Metadata JSON
meta = {
    'sample_fraction': SAMPLE_FRAC,
    'train_rows': int(len(X_train_sc)),
    'test_rows' : int(len(X_test_sc)),
    'features'  : len(FEATURE_NAMES),
    'GB_tuned': {
        'accuracy' : round(float(acc),  4),
        'precision': round(float(prec), 4),
        'recall'   : round(float(rec),  4),
        'f1'       : round(float(f1),   4),
        'roc_auc'  : round(float(auc),  4),
        'mcc'      : round(float(mcc),  4),
        'best_params': rs.best_params_
    },
    'RF': {
        'f1'     : round(float(f1_score(y_test, y_pred_rf)), 4),
        'roc_auc': round(float(roc_auc_score(y_test, y_proba_rf)), 4)
    }
}
with open(OUT/'metadata.json','w') as fh:
    json.dump(meta, fh, indent=2)

print("✓ Saved to outputs/")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")


✓ Saved to outputs/
  feature_importance.csv  (2.9 KB)
  feature_names.pkl  (1.5 KB)
  gb_final.pkl  (500.0 KB)
  metadata.json  (0.5 KB)
  rf_model.pkl  (1623.2 KB)
  scaler.pkl  (2.4 KB)


## 15 · Summary & Next Steps

In [34]:
print("""
╔══════════════════════════════════════════════════════╗
║          PHASE 1 COMPLETE — RESULTS SUMMARY          ║
╠══════════════════════════════════════════════════════╣
║  Model                F1-Score   ROC-AUC             ║""")

rf_f1  = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_proba_rf)
gb_f1  = f1_score(y_test, y_pred_gb)
gb_auc = roc_auc_score(y_test, y_proba_gb)

print(f"""║  Random Forest        {rf_f1:.4f}     {rf_auc:.4f}             ║
║  GB Baseline          {gb_f1:.4f}     {gb_auc:.4f}             ║
║  GB Tuned (FINAL)     {f1:.4f}     {auc:.4f}             ║
╠══════════════════════════════════════════════════════╣
║  NEXT → Phase 2: test same models on NSL-KDD         ║
║         to measure concept-drift / adaptability      ║
╚══════════════════════════════════════════════════════╝""")



╔══════════════════════════════════════════════════════╗
║          PHASE 1 COMPLETE — RESULTS SUMMARY          ║
╠══════════════════════════════════════════════════════╣
║  Model                F1-Score   ROC-AUC             ║
║  Random Forest        0.9993     1.0000             ║
║  GB Baseline          0.9997     1.0000             ║
║  GB Tuned (FINAL)     0.9996     1.0000             ║
╠══════════════════════════════════════════════════════╣
║  NEXT → Phase 2: test same models on NSL-KDD         ║
║         to measure concept-drift / adaptability      ║
╚══════════════════════════════════════════════════════╝
